# Nebius x Prismatic Waste Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prismatic-labs/vetch/blob/main/demos/nebius_vetch_waste_lab_colab.ipynb)

Metadata-only inference waste observability on top of Nebius Token Factory, powered by Vetch.

Runs four real Nebius calls — baseline, RAG bloat, agent loop, and JSON retries — then applies Vetch-style waste detection. Requires a Nebius API key.

**Privacy boundary:** no prompts, completions, raw payloads, tool arguments, user secrets, or API keys are stored. Derived metadata only.


## What it shows

- **Baseline**: compact, low-latency calls with no waste flags.
- **RAG bloat**: large retrieval context that inflates input tokens while the answer stays small.
- **Agent loop**: repeated tool calls accumulating latency and tokens across turns.
- **JSON retries**: extra tokens burned before the model produces valid structured output.

Adjust `N_PER_SCENARIO` in the setup cell to control credit spend.


In [ ]:
%pip -q install pandas matplotlib pydantic openai


In [ ]:
from __future__ import annotations
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

from collections import Counter, defaultdict
from datetime import datetime, timezone
from statistics import median
from pathlib import Path
from typing import Optional, Literal
import json, html, uuid, time

import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class HTML(str): pass
    class Markdown(str): pass
    def display(v): print(v)
from pydantic import BaseModel, Field


class InferenceEvent(BaseModel):
    run_id: str
    scenario: str
    provider: Literal['nebius'] = 'nebius'
    model: str
    started_at: datetime
    ended_at: datetime
    latency_ms: int
    input_tokens: Optional[int] = None
    output_tokens: Optional[int] = None
    total_tokens: Optional[int] = None
    retry_count: int = 0
    tool_call_count: int = 0
    cache_candidate: bool = False
    cacheability_score: float = Field(default=0.0, ge=0.0, le=1.0)
    waste_flags: list[str] = Field(default_factory=list)
    waste_score: float = 0.0
    notes: list[str] = Field(default_factory=list)

    def safe_dict(self) -> dict:
        return self.model_dump(mode='json')

InferenceEvent.model_rebuild()

NEBIUS_BASE_URL   = 'https://api.tokenfactory.nebius.com/v1/'
N_PER_SCENARIO    = 2
SCENARIOS         = ['baseline', 'rag_bloat', 'agent_loop', 'json_retries']
HIGH_LATENCY_MS   = 2500
TOOL_CALL_THRESHOLD = 5
FLAG_WEIGHTS = {
    'RAG_BLOAT':        0.35,
    'AGENT_LOOP':       0.35,
    'JSON_RETRY_WASTE': 0.25,
    'CACHE_OPPORTUNITY':0.20,
    'HIGH_LATENCY':     0.15,
}


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def usage_value(usage, *names):
    for name in names:
        v = getattr(usage, name, None)
        if v is not None:
            return int(v)
    return None


def apply_rules(
    event: InferenceEvent,
    baseline_input_median: Optional[float] = None,
    baseline_output_median: Optional[float] = None,
) -> InferenceEvent:
    flags = set(event.waste_flags)
    notes = list(event.notes)

    if event.scenario == 'rag_bloat' and baseline_input_median and event.input_tokens:
        comparable_out = (event.output_tokens or 0) <= (baseline_output_median or 1) * 1.5
        if event.input_tokens > baseline_input_median * 3 and comparable_out:
            flags.add('RAG_BLOAT')
            notes.append('Input tokens exceed 3x baseline while output stays comparable.')

    if event.tool_call_count > TOOL_CALL_THRESHOLD:
        flags.add('AGENT_LOOP')
        notes.append('Tool calls exceed the configured threshold for one task.')

    if event.retry_count > 0:
        flags.add('JSON_RETRY_WASTE')
        notes.append('Retries consumed extra tokens before success.')

    if event.cache_candidate and event.cacheability_score >= 0.75 and (event.input_tokens or 0) > 500:
        flags.add('CACHE_OPPORTUNITY')
        notes.append('Stable task prefix looks cacheable across repeated calls.')

    if event.latency_ms > HIGH_LATENCY_MS:
        flags.add('HIGH_LATENCY')
        notes.append('Latency exceeded configured threshold.')

    event.waste_flags = sorted(flags)
    event.waste_score = round(min(1.0, sum(FLAG_WEIGHTS.get(f, 0.1) for f in flags)), 3)
    event.notes = sorted(set(notes))
    return event


In [ ]:
from getpass import getpass
from openai import OpenAI

_key = getpass('Paste NEBIUS_API_KEY (hidden, not stored): ')
client = OpenAI(api_key=_key, base_url=NEBIUS_BASE_URL)
del _key

available = [m.id for m in client.models.list().data]
NEBIUS_MODEL = available[0]

display(HTML(f"""
<div style='font-family:system-ui;padding:14px 16px;border-left:4px solid #81B29A;
             background:#f3faf6;border-radius:0 12px 12px 0'>
  <b>Connected.</b> Using: <code>{html.escape(NEBIUS_MODEL)}</code><br>
  <span style='color:#4d6257;font-size:13px'>All available: {html.escape(', '.join(available))}</span>
</div>
"""))


In [ ]:
run_id = uuid.uuid4().hex[:12]
events: list[InferenceEvent] = []

# baseline
print('baseline...')
for i in range(N_PER_SCENARIO):
    msgs = [
        {'role': 'system', 'content': 'Return compact operational advice.'},
        {'role': 'user',   'content': f'Summarise a deployment note in 3 bullets. Run {i+1}.'},
    ]
    t0 = utc_now(); s = time.perf_counter()
    r = client.chat.completions.create(model=NEBIUS_MODEL, messages=msgs, temperature=0.2, max_tokens=140)
    lms = int((time.perf_counter()-s)*1000)
    u = r.usage
    events.append(apply_rules(InferenceEvent(
        run_id=run_id, scenario='baseline', model=NEBIUS_MODEL,
        started_at=t0, ended_at=utc_now(), latency_ms=lms,
        input_tokens=usage_value(u,'prompt_tokens','input_tokens'),
        output_tokens=usage_value(u,'completion_tokens','output_tokens'),
        total_tokens=usage_value(u,'total_tokens'),
        cache_candidate=False, cacheability_score=0.2,
        notes=['Compact task. No prompt or completion text stored.'],
    )))

# rag_bloat
print('rag_bloat...')
ctx = '\n'.join(f'Retrieved chunk {j}: deployment policy detail repeated for stress testing.' for j in range(80))
for i in range(N_PER_SCENARIO):
    msgs = [
        {'role': 'system', 'content': 'Answer from the provided context. Be concise.'},
        {'role': 'user',   'content': f'{ctx}\n\nQuestion: what is the deployment risk?'},
    ]
    t0 = utc_now(); s = time.perf_counter()
    r = client.chat.completions.create(model=NEBIUS_MODEL, messages=msgs, temperature=0.2, max_tokens=160)
    lms = int((time.perf_counter()-s)*1000)
    u = r.usage
    events.append(apply_rules(InferenceEvent(
        run_id=run_id, scenario='rag_bloat', model=NEBIUS_MODEL,
        started_at=t0, ended_at=utc_now(), latency_ms=lms,
        input_tokens=usage_value(u,'prompt_tokens','input_tokens'),
        output_tokens=usage_value(u,'completion_tokens','output_tokens'),
        total_tokens=usage_value(u,'total_tokens'),
        cache_candidate=True, cacheability_score=0.86,
        notes=['Oversized retrieval context. Token counts only stored.'],
    )))

# agent_loop
print('agent_loop...')
tools = [{'type':'function','function':{
    'name':'lookup_metric',
    'description':'Look up a synthetic deployment metric.',
    'parameters':{'type':'object','properties':{'metric':{'type':'string'}},'required':['metric']},
}}]
for i in range(N_PER_SCENARIO):
    conv = [
        {'role':'system','content':'Use tools if needed. Avoid repeated lookups.'},
        {'role':'user',  'content':'Investigate the missing deployment metric until confident.'},
    ]
    t0 = utc_now(); s = time.perf_counter()
    tin = tout = ncalls = 0
    for _ in range(TOOL_CALL_THRESHOLD + 2):
        try:
            r = client.chat.completions.create(model=NEBIUS_MODEL, messages=conv, temperature=0.1, max_tokens=200, tools=tools)
        except Exception:
            r = client.chat.completions.create(model=NEBIUS_MODEL, messages=conv, temperature=0.1, max_tokens=200)
            break
        u = r.usage
        if u:
            tin  += usage_value(u,'prompt_tokens','input_tokens') or 0
            tout += usage_value(u,'completion_tokens','output_tokens') or 0
        ch = r.choices[0]
        if ch.finish_reason != 'tool_calls' or not getattr(ch.message,'tool_calls',None):
            break
        ncalls += len(ch.message.tool_calls)
        conv.append(ch.message)
        for tc in ch.message.tool_calls:
            conv.append({'role':'tool','tool_call_id':tc.id,'content':'metric_value: unavailable'})
    lms = int((time.perf_counter()-s)*1000)
    events.append(apply_rules(InferenceEvent(
        run_id=run_id, scenario='agent_loop', model=NEBIUS_MODEL,
        started_at=t0, ended_at=utc_now(), latency_ms=lms,
        input_tokens=tin or None, output_tokens=tout or None,
        total_tokens=(tin+tout) or None, tool_call_count=ncalls,
        cache_candidate=True, cacheability_score=0.72,
        notes=[f'{ncalls} tool call(s). Tool arguments not stored.'],
    )))

# json_retries
print('json_retries...')
for i in range(N_PER_SCENARIO):
    msgs = [
        {'role':'system','content':'Return strict JSON with keys "status" and "risk". Output JSON only, no prose.'},
        {'role':'user',  'content':'Classify this deployment change as JSON only.'},
    ]
    t0 = utc_now(); s = time.perf_counter()
    tin = tout = retries = 0
    for attempt in range(4):
        r = client.chat.completions.create(model=NEBIUS_MODEL, messages=msgs, temperature=0.9, max_tokens=80)
        u = r.usage
        if u:
            tin  += usage_value(u,'prompt_tokens','input_tokens') or 0
            tout += usage_value(u,'completion_tokens','output_tokens') or 0
        content = r.choices[0].message.content or ''
        try:
            json.loads(content); break
        except json.JSONDecodeError:
            retries += 1
            msgs = msgs + [
                {'role':'assistant','content':content},
                {'role':'user','content':'Invalid JSON. Return only a JSON object with keys status and risk.'},
            ]
    lms = int((time.perf_counter()-s)*1000)
    events.append(apply_rules(InferenceEvent(
        run_id=run_id, scenario='json_retries', model=NEBIUS_MODEL,
        started_at=t0, ended_at=utc_now(), latency_ms=lms,
        input_tokens=tin or None, output_tokens=tout or None,
        total_tokens=(tin+tout) or None, retry_count=retries,
        cache_candidate=True, cacheability_score=0.82,
        notes=[f'{retries} retry/retries before valid JSON. Prompt and completion text not stored.'],
    )))

# re-apply rules with baseline medians now that all events exist
bl = [e for e in events if e.scenario == 'baseline']
if bl:
    b_in  = median([e.input_tokens  for e in bl if e.input_tokens])
    b_out = median([e.output_tokens for e in bl if e.output_tokens])
    events = [apply_rules(e, b_in, b_out) for e in events]

df = pd.DataFrame([e.safe_dict() for e in events])
total_tokens_obs = int(df['total_tokens'].sum())
waste_count      = sum(1 for f in df['waste_flags'] if f)
max_latency_obs  = int(df['latency_ms'].max())
print(f'Done. {len(events)} events | {total_tokens_obs:,} tokens | {waste_count} flagged')


In [ ]:
FORBIDDEN = {'prompt','completion','messages','raw_request','raw_response','api_key','tool_args'}
bad = sorted(set(df.columns) & FORBIDDEN)
if bad:
    display(HTML(f"<div style='padding:16px;border-left:5px solid #d33;background:#fff1f1'><b>Privacy check failed:</b> {bad}</div>"))
else:
    display(HTML("""
    <div style='font-family:system-ui;display:grid;grid-template-columns:auto 1fr;gap:16px;
                align-items:center;padding:18px 20px;border:1px solid #cfe9da;background:#f3fbf6;
                border-radius:14px;box-shadow:0 12px 34px rgba(27,143,90,.08)'>
      <div style='width:48px;height:48px;border-radius:14px;background:#123d29;color:#bff0d2;
                  display:grid;place-items:center;font-size:24px;font-weight:950'>&#10003;</div>
      <div>
        <div style='font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#1b8f5a;font-weight:900'>Privacy check passed</div>
        <div style='font-size:22px;font-weight:950;margin-top:3px;color:#13251a'>Telemetry contains metadata only.</div>
        <div style='margin-top:5px;color:#4d6257'>No prompt text, completion text, raw payloads, tool arguments, API keys, or user secrets are stored.</div>
      </div>
    </div>
    """))


In [ ]:
rec = events[0].safe_dict()
rows = []
for k, v in rec.items():
    rows.append(f'  <span style="color:#81B29A;font-weight:700">"{k}"</span>: '
                f'<span style="color:#F2CC8F">{html.escape(json.dumps(v, default=str))}</span>')
NL = chr(10)
display(HTML(f"""
<div style='font-family:system-ui;margin-top:14px'>
  <div style='font-size:12px;text-transform:uppercase;letter-spacing:.12em;color:#81B29A;
              font-weight:950;margin-bottom:10px'>One stored event</div>
  <pre style='font-family:Menlo,Monaco,Consolas,monospace;background:#0b0d0c;color:#edf5f0;
              border:1px solid #26382f;border-radius:14px;padding:20px 24px;
              overflow-x:auto;font-size:13px;line-height:1.75;margin:0'>{{{NL}{NL.join(rows)}{NL}}}</pre>
  <div style='margin-top:10px;font-size:13px;color:#8fa098'>No <code>prompt</code>, <code>completion</code>,
  <code>messages</code>, <code>raw_request</code>, <code>raw_response</code>, <code>tool_args</code>,
  or <code>api_key</code> fields.</div>
</div>
"""))


In [ ]:
FLAG_COLORS = {
    'RAG_BLOAT':         '#f0b35a',
    'AGENT_LOOP':        '#ff6b87',
    'JSON_RETRY_WASTE':  '#a99cff',
    'CACHE_OPPORTUNITY': '#81B29A',
    'HIGH_LATENCY':      '#F2CC8F',
}
SCENARIO_STORIES = {
    'baseline':    'Compact work. Low latency. No control action needed.',
    'rag_bloat':   'The answer stays small while retrieved context inflates the bill.',
    'agent_loop':  'Tool turns repeat past the useful point.',
    'json_retries':'Invalid structured output creates avoidable retry spend.',
}

def pill(flag):
    c = FLAG_COLORS.get(flag,'#777')
    return f"<span style='display:inline-block;background:rgba(255,255,255,.045);border:1px solid {c};"\
           f"border-radius:999px;padding:4px 8px;font-size:10px;font-weight:900;margin:0 5px 6px 0;"\
           f"letter-spacing:.03em;color:{c}'>{html.escape(flag)}</span>"

groups = defaultdict(list)
for e in events:
    groups[e.scenario].append(e)
max_tok = max((e.total_tokens or 1) for e in events)
max_lat = max(e.latency_ms for e in events)
cards = []
for sc in SCENARIOS:
    rows = groups[sc]
    avg_in  = sum(e.input_tokens  or 0 for e in rows)/len(rows)
    avg_out = sum(e.output_tokens or 0 for e in rows)/len(rows)
    avg_lat = sum(e.latency_ms        for e in rows)/len(rows)
    avg_wst = sum(e.waste_score       for e in rows)/len(rows)
    flags   = sorted({f for e in rows for f in e.waste_flags})
    fpills  = ' '.join(pill(f) for f in flags) or "<span style='color:#8fa098;font-size:12px'>clean trace</span>"
    bars    = []
    for idx,e in enumerate(rows,1):
        tw = max(4,int(((e.total_tokens or 0)/max_tok)*100))
        lw = max(4,int((e.latency_ms/max_lat)*100))
        bars.append(f"<div style='display:grid;grid-template-columns:30px 1fr 88px 82px;gap:8px;"
                    f"align-items:center;font-size:12px;color:#c9d4ce'>"
                    f"<span>#{idx}</span><div style='display:grid;gap:3px'>"
                    f"<i style='display:block;height:7px;border-radius:999px;width:{tw}%;"
                    f"background:linear-gradient(90deg,#81B29A,#4A9A8C)'></i>"
                    f"<i style='display:block;height:7px;border-radius:999px;width:{lw}%;"
                    f"background:linear-gradient(90deg,#F2CC8F,#E07A5F)'></i></div>"
                    f"<b style='color:#dfe8e3'>{e.total_tokens:,} tok</b>"
                    f"<em style='font-style:normal;color:#dfe8e3'>{e.latency_ms:,} ms</em></div>")
    ring = int(avg_wst*100)
    cards.append(f"""
    <section style='position:relative;overflow:hidden;border:1px solid #24342b;border-radius:18px;
                    padding:20px;background:#101311;color:#eef5f0;box-shadow:0 18px 50px rgba(0,0,0,.16)'>
      <div style='display:flex;justify-content:space-between;gap:16px;align-items:flex-start;margin-bottom:14px'>
        <div>
          <div style='font-size:12px;text-transform:uppercase;letter-spacing:.14em;font-weight:950;
                      color:#9ed0b7;margin-bottom:6px'>{html.escape(sc.replace('_',' '))}</div>
          <p style='margin:0;color:#aab8b0;line-height:1.45;font-size:14px'>{html.escape(SCENARIO_STORIES[sc])}</p>
        </div>
        <div style='width:70px;height:70px;border-radius:999px;display:grid;place-items:center;
                    background:conic-gradient(#E07A5F {ring}%,rgba(255,255,255,.08) 0);
                    box-shadow:inset 0 0 0 8px #111713;flex:0 0 auto'>
          <b style='font-size:18px'>{avg_wst:.2f}</b>
        </div>
      </div>
      <div style='display:grid;grid-template-columns:repeat(3,1fr);gap:9px;margin-bottom:12px'>
        <div style='background:rgba(255,255,255,.06);border:1px solid rgba(255,255,255,.09);border-radius:12px;padding:10px'>
          <b style='display:block;font-size:18px;color:#fff'>{avg_in:,.0f}</b>
          <span style='font-size:11px;color:#8fa098;text-transform:uppercase;letter-spacing:.08em'>avg input</span></div>
        <div style='background:rgba(255,255,255,.06);border:1px solid rgba(255,255,255,.09);border-radius:12px;padding:10px'>
          <b style='display:block;font-size:18px;color:#fff'>{avg_out:,.0f}</b>
          <span style='font-size:11px;color:#8fa098;text-transform:uppercase;letter-spacing:.08em'>avg output</span></div>
        <div style='background:rgba(255,255,255,.06);border:1px solid rgba(255,255,255,.09);border-radius:12px;padding:10px'>
          <b style='display:block;font-size:18px;color:#fff'>{avg_lat:,.0f} ms</b>
          <span style='font-size:11px;color:#8fa098;text-transform:uppercase;letter-spacing:.08em'>avg latency</span></div>
      </div>
      <div style='margin-bottom:10px'>{fpills}</div>
      <div style='display:grid;gap:9px'>{''.join(bars)}</div>
    </section>""")

display(HTML(f"<div style='display:grid;grid-template-columns:repeat(2,minmax(0,1fr));"
             f"gap:18px;font-family:system-ui;margin-top:14px'>{''.join(cards)}</div>"))


In [ ]:
summary = (
    df.groupby('scenario')
    .agg(
        events=('run_id','count'),
        avg_input_tokens=('input_tokens','mean'),
        avg_output_tokens=('output_tokens','mean'),
        avg_latency_ms=('latency_ms','mean'),
        avg_waste_score=('waste_score','mean'),
        retries=('retry_count','sum'),
        tool_calls=('tool_call_count','sum'),
    )
    .reindex(SCENARIOS)
)
pretty = summary.copy()
for c in ['avg_input_tokens','avg_output_tokens','avg_latency_ms']:
    pretty[c] = pretty[c].round(0).astype(int)
pretty['avg_waste_score'] = pretty['avg_waste_score'].round(2)
display(pretty)

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(17, 4.7))
fig.patch.set_facecolor('#0b0d0c')
for ax in axes:
    ax.set_facecolor('#101311')
    ax.grid(axis='y', color='#26352d', linewidth=.8, alpha=.75)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color('#34483d')
    ax.spines['bottom'].set_color('#34483d')
    ax.tick_params(colors='#b8c6bf')
summary[['avg_input_tokens','avg_output_tokens']].plot(kind='bar', ax=axes[0], color=['#81B29A','#F2CC8F'], width=.72)
axes[0].set_title('Token shape', color='#f2f7f4', fontweight='bold'); axes[0].set_xlabel('')
summary['avg_latency_ms'].plot(kind='bar', ax=axes[1], color='#4A9A8C', width=.72)
axes[1].set_title('Latency pressure', color='#f2f7f4', fontweight='bold'); axes[1].set_xlabel('')
summary['avg_waste_score'].plot(kind='bar', ax=axes[2], color='#E07A5F', width=.72)
axes[2].set_title('Waste signal', color='#f2f7f4', fontweight='bold')
axes[2].set_xlabel(''); axes[2].set_ylim(0,1)
axes[2].axhline(.5, color='#F2CC8F', linestyle='--', linewidth=1, alpha=.8)
plt.tight_layout(); plt.show()


In [ ]:
flag_counts = Counter(f for flags in df['waste_flags'] for f in flags)

def avoidable_tokens(event: InferenceEvent, baseline_input: float) -> int:
    tin  = event.input_tokens  or 0
    tout = event.output_tokens or 0
    av   = 0
    if 'RAG_BLOAT'        in event.waste_flags: av += max(0, int(tin - baseline_input * 2))
    if 'JSON_RETRY_WASTE' in event.waste_flags: av += event.retry_count * max(1, int((tin+tout)/(event.retry_count+1)))
    if 'AGENT_LOOP'       in event.waste_flags: av += max(0, event.tool_call_count - TOOL_CALL_THRESHOLD) * 150
    if 'CACHE_OPPORTUNITY'in event.waste_flags: av += int(tin * 0.25)
    return av

b_in_med = median([e.input_tokens for e in events if e.scenario=='baseline' and e.input_tokens])
avoidable = sum(avoidable_tokens(e, b_in_med) for e in events)

def md_table(frame):
    lines = [
        '| Scenario | Events | Avg input | Avg output | Avg latency ms | Avg waste | Retries | Tool calls |',
        '|---|---:|---:|---:|---:|---:|---:|---:|',
    ]
    for sc, row in frame.iterrows():
        lines.append(f'| {sc} | {int(row.events)} | {row.avg_input_tokens:,.0f} | {row.avg_output_tokens:,.0f} |'
                     f' {row.avg_latency_ms:,.0f} | {row.avg_waste_score:.2f} | {int(row.retries)} | {int(row.tool_calls)} |')
    return chr(10).join(lines)

report = f'''# Nebius Token Factory Waste Lab

Model: {NEBIUS_MODEL}  
Run ID: {run_id}

## Summary

- Total tokens observed: {total_tokens_obs:,}
- Estimated avoidable tokens: {avoidable:,}
- Events with waste flags: {waste_count} of {len(events)}

## Scenario comparison

{md_table(summary)}

## Waste flags

{chr(10).join(f"- {f}: {c} event(s)" for f, c in sorted(flag_counts.items())) or "- none triggered"}

## What Nebius could expose to sharpen the signal

- Request ID and region
- Endpoint type: serverless vs dedicated
- GPU type
- Time to first token and tokens/sec
- Queue time
- Cache hit/miss
- Per-request usage metadata
- Project/team/customer attribution fields

## Privacy boundary

No prompts, completions, raw request/response bodies, tool arguments, API keys, or user secrets were stored.
'''

out_dir = Path('/content') if Path('/content').exists() else Path.cwd() / 'reports'
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / 'nebius_waste_report.md'
path.write_text(report, encoding='utf-8')

display(Markdown(report))
display(HTML(f"<div style='padding:12px 14px;border-left:4px solid #81B29A;background:#f3faf6;"
             f"font-family:system-ui'><b>Report written:</b> {path}</div>"))
